## Why Gradients Matter

### What is a gradient?
- A gradient is a vector of partial derivatives that measures how the loss function (error) changes with respect to small changes in each model parameter (weights and biases).  It indicates the direction of the steepest ascent in the error landscape, allowing algorithms to determine how to adjust parameters to minimize prediction errors.

- You can think of it like a hill:

        Loss = your height on the hill
        Gradient = slope direction + steepness

        If slope is steep:

                large gradient
                big updates

        If slope is flat:

                small gradient
                tiny updates


        At the minimum:

                ∇L≈0

                because the surface becomes flat near the bottom.

### Why do neural networks need gradients?
- Neural networks need gradients because they provide the direction and magnitude required to adjust model parameters (weights and biases) in order to minimize the loss function.  Without gradients, the network cannot determine how to reduce the error between its predictions and the actual target values, making learning from data impossible. 

### What does positive gradient and negative gradient imply?
- positive gradient implies that increasing weight or bias leads to increase in loss.
- negative gradient implies that increasing weight or bias leads to decrease in loss.

- That is why gradient decent moves weight opposite to gradient direction thus decreasing loss. 

## Tensor Gradient Tracking

In [19]:
import torch

x = torch.tensor(4.0, requires_grad=True)
x

tensor(4., requires_grad=True)

### What does requires_grad=True do?
- It tells Pytorch to record operations step-by-step. Pytorch internally computes graph of what happens to tensor.

### Why are gradients not tracked by default?
- Because it is expensive to track gradient. Pytorch needs to store every operation, intermediate value and entire computation graph.

## Build Simple Computation Graph

In [20]:
y = x**2 + 2*x + 1
y

tensor(25., grad_fn=<AddBackward0>)

### What operations were applied to x?
- x -> square -> multiply -> add -> y
- Square, multiplication and addition is applied to x.

### Why does PyTorch store this computation history?
- Because we told to by setting "requires_grad=True".

## Backpropagation Basics

In [21]:
y.backward()
x.grad

tensor(10.)

### What value do you get?
- We get 10.0 as the gradient of y with respect to x when x = 4.0 and y = x**2 + 2*x + 1.

### What mathematical derivative is PyTorch computing?
- PyTorch is computing the derivative of y with respect to x using automatic differentiation, i.e. dy/dx.
For the function y = x**2 + 2*x + 1, the derivative is 2x + 2. At x = 4, the gradient becomes 10.

- "PyTorch computes how y changes with respect to x."

## Gradient Interpretation

In [22]:
x.grad

tensor(10.)

### What does this mean geometrically?
- Geometrically, the gradient represents the slope of the curve y = x**2 + 2*x + 1 at x = 4. It tells us how rapidly y changes for a small change in x near that point.

### Why would optimization move in the opposite direction
- Optimization moves in the opposite direction of the gradient because the gradient points toward the direction of greatest increase. 
- Since training aims to minimize loss (error), we move in the negative gradient direction to reduce the loss.

## Multiple Variables

In [23]:
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

z = a*b + b**2

In [24]:
z.backward()

In [25]:
print(a.grad)
print(b.grad)

tensor(4.)
tensor(11.)


### What is: ∂z/∂a ?
- Since z contains two variable we need to partially differentiate to get gradient. So gradient of z with respect to a is 4.0 at a = 3.0 and b = 4.0 which means when a changes by a very small amount near a=3, z changes approximately 4 times that amount.

### What is: ∂z/∂b ?
- Similarly, we need partially differentiate and it gives gradient of z with respect to b as 11.0 at a = 3.0 and b = 4.0 which means when a changes by a very small amount near b=4, z changes approximately 11 times that amount.

### Manual Verification
- As a = 3.0, b = 4.0 and z = a*b + b**2 :

        ∂z/∂a = b + 0 = 4.0

        ∂z/∂b = a + 2b = 3.0 + 2(4.0) = 11.0

## Gradient Accumulation

In [26]:
z = a*b + b**2
z.backward()

print(a.grad)
print(b.grad)

tensor(8.)
tensor(22.)


### Why did gradients increase?
- Gradients increased because PyTorch accumulates gradients in the `.grad` attribute instead of replacing them. Each call to `backward()` adds newly computed gradients to the existing stored gradients.

### Why does PyTorch accumulate gradients instead of replacing them?
- Because in real neural network training sometimes gradients from multiple computations or batches need to be accumulated before updating parameters. So accumulation is useful and intentional.

## Resetting Gradients

In [27]:
a.grad.zero_()
b.grad.zero_()

print(a.grad)
print(b.grad)

tensor(0.)
tensor(0.)


### Why must gradients be reset during training loops?
- Gradients must be reset during training loops because PyTorch accumulates gradients by default.
- After each parameter update, old gradients are no longer valid since the model weights have already changed.
- If gradients are not cleared, gradients from previous batches keep accumulating, leading to incorrect parameter updates and potentially unstable training.

## Disable Gradient Tracking

In [31]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x * 2
    
print(x.grad)

None


### Why disable gradients during inference?
- Gradients are disabled during inference because the model is not being trained, so gradient computation is unnecessary.
- Using `torch.no_grad()` makes inference faster and more memory-efficient.

### Why would tracking gradients during inference waste memory?
- Tracking gradients would force PyTorch to build computational graphs and store intermediate values, which increases memory usage and computation time.

## Computational Graph Intuition

### What is a computational graph?
- A computational graph is a structure that records tensors and the operations connecting them. It allows PyTorch to track how outputs depend on inputs and parameters.

### Why is deep learning fundamentally: graph construction + automatic differentiation
- Deep learning is fundamentally graph construction plus automatic differentiation because neural network training requires:

    1. building a chain of computations during the forward pass (graph computation)
    2. automatically computing gradients through that chain during the backward pass (automatic differentiation)

- Steps taken during training:

    Step 1 — Graph Construction

        Forward pass:

            input → layers → activations → prediction → loss

            creates a huge computational graph.

            Every operation becomes part of the graph.

    Step 2 — Automatic Differentiation

        Backward pass:

            loss.backward()

            uses the graph to apply:

            chain rule automatically

            and compute gradients for every parameter.

## Vector Gradients

In [33]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = (x**2).sum()

y.backward()

x.grad

tensor([2., 4., 6.])

### Why was .sum() necessary?
- We need to convert y which is in vector form due to x being a vector to scaler because .backward() excepts scaler as output and .sum() converts vector to scaler (single value) by adding all the outputs.
- Now Pytorch can compute how final output (scaler) depends on x and optimize it.

### What happens if backward is called on non-scalar output?
- Pytorch raises an error because the gradient of a vector output is ambiguous without additional information. we need to provide an external gradient argument so that Pytorch can determine the backward computation properly.
- Note: For scaler output this is automatic.

## Relation to Deep Learning

### In neural networks, what tensors usually require gradients?
- In neural network tensors representing learnable parameter usually requires gradient. because during training we want to learn how changing these parameters affects the loss.
- Gradients allow optimization algorithms like gradient descent to update them.
- Parameters such as weights and biases.

### Why do input tensors usually NOT require gradients?
- Input tensors usually do not require gradients because inputs are not being optimized.
- During training, the goal is adjust model parameters, not modify the training data.
- So gradients for inputs are unnecessary and would waste memory and computation.